#**Character RNN 실습자료**
Practice, Char-level vanilla RNN model
References:
1. https://gist.github.com/karpathy/d4dee566867f8291f086
2. https://github.com/javaidnabi31/RNN-from-scratch


###모델 학습에 대한 전반적인 이해 순서

1. 모델 학습을 위한 Datareader 클래스 생성 및 호출
2. 모델 클래스 설계 및 모델 학습
3. 학습된 모델을 통한 추론 과정 설계 및 테스트

###세부 순서
1. 가중치 행렬(weight matrics) U, V, W 와 편향(bias) b, c를 0으로 초기화
2. 모델 학습을 위한 순방향 전파(forward propagation)
3. 손실율(정답값 - 예측값) 계산
4. 기울기(Gradient) 계산을 위한 역전파(backward propagation)
5. 기울기(Gradient)에 기반한 weight 업데이트
6. 2-5 과정 반복

### Input.txt 파일 저장하기

###기초 라이브러리 함수 호출

In [ ]:
import numpy as np

###학습 데이터 읽기 & 문자 인덱싱할 어휘 및 사전 생성

In [ ]:
class DataReader:
    def __init__(self, path, seq_length): # __init__: 텍스트 파일을 읽어서 앞 100자만 사용, 등장하는 고유 문자들로 문자 <-> 인덱스 매핑 딕셔너리 생성 / 객체 만들 때 path(텍스트 파일 경로)랑 seq_length(한 번에 학습시킬 시퀀스 길이)를 받음.

        """
        self.data = "We know what we are, but know not what we may be. " \
                    "Modest doubt is called the beacon of the wise. " \
                    "All that glisters is not gold."
        """

        with open(path, "r") as fp: # path에 있는 텍스트 파일을 열어서 전체 내용을 문자열로 읽어 self.data에 저장.
            self.data= fp.read()
        self.data = self.data[:100] # 읽은 텍스트 중 앞 100글자만 잘라서 씀.

        # find unique chars
        chars = list(set(self.data)) # self.data에 등장하는 중복 없는 문자들의 리스트를 만듦. set()으로 중복 제거하고 list()로 순서 있는 자료구조로 전환.

        #create dictionary mapping for each char
        self.char_to_ix = {ch:i for (i,ch) in enumerate(chars)} # 문자 -> 인덱스 매핑 딕셔너리.
        self.ix_to_char = {i:ch for (i,ch) in enumerate(chars)} # 인덱스 -> 문자. 모델이 예측한 숫자를 다시 문자로 바꿀 때 씀.
        # enumerate(chars)가 (0,chars[0]),(1,chars[1]),..이런식으로 인덱스랑 값을 같이 던짐. 딕셔너리 컴프리헨션으로 한 줄에 매핑 완성.

        print("self.char_to_ix:", self.char_to_ix)
        print("self.ix_to_char:", self.ix_to_char)

        #total data
        self.data_size = len(self.data) # 전체 텍스트 길이

        # num of unique chars
        self.vocab_size = len(chars) # 고유 문자 개수(알파벳/공백/마침표 등 종류 수). 모델의 입력/출력 차원(원-핫 벡터 크기)으로 쓰임.
        self.pointer = 0 # pointer는 지금 데이터의 어디까지 읽었는지 위치를 기억하는 커서. 처음엔 0에서 시작.
        self.seq_length = seq_length # seq_length는 받아온 값 저장.

    def next_batch(self): # seq_length 길이만큼 잘라서 input/target 시퀀스 반환(target은 input에서 한 칸 밀린 것 - 다음 문자 예측용)
        input_start = self.pointer # 현재 포인터 위치부터 seq_length만큼 뒤까지를 이번 배치 범위로 잡음.
        input_end = self.pointer + self.seq_length
        inputs = [self.char_to_ix[ch] for ch in self.data[input_start:input_end]] # self.data에서 [input_start:input_end]구간의 문자들을 하나씩 꺼내서, char_to_ix로 정수 인덱스로 변환한 리스트. 이게 입력 시퀀스
        targets = [self.char_to_ix[ch] for ch in self.data[input_start + 1:input_end + 1]] # 입력에서 한 칸씩 밀린 구간을 타겟으로 씀. char-RNN은 "다음 글자 맞추기"가 목표라서, 입력이 "hell"이면 타켓은 "ello"가 되는 식.
        self.pointer += self.seq_length                         # pointer 위치를 변경해가며 다음 batch에 해당하는 부분 읽기(다음 배치를 위해 포인터를 seq_length만큼 전진.)
        if self.pointer + self.seq_length + 1 >= self.data_size: # 다음번에 뽑을 배치가 데이터 끝을 넘어갈 것 같으면 포인터를 0으로 리셋.
            self.pointer = 0
        return inputs, targets # 정수 인덱스로 이루어진 입력/타겟 시퀀스 쌍을 반환.

    def just_started(self): # pointer가 0인지 (즉 한 바퀴 다 돌았는지)체크
        return self.pointer == 0

In [ ]:
# 데이터 양 체크
check_data = DataReader("input.txt", seq_length=50)

print("==============================")
print(check_data.data)
print("==============================")

self.char_to_ix: {'F': 0, 's': 1, 't': 2, 'h': 3, 'm': 4, 'r': 5, ',': 6, 'c': 7, ':': 8, 'f': 9, 'l': 10, 'Y': 11, 'i': 12, 'a': 13, 'o': 14, 'A': 15, 'n': 16, 'e': 17, 'y': 18, 'C': 19, '\n': 20, 'd': 21, 'S': 22, '.': 23, 'p': 24, ' ': 25, 'w': 26, 'u': 27, 'z': 28, 'k': 29, 'B': 30}
self.ix_to_char: {0: 'F', 1: 's', 2: 't', 3: 'h', 4: 'm', 5: 'r', 6: ',', 7: 'c', 8: ':', 9: 'f', 10: 'l', 11: 'Y', 12: 'i', 13: 'a', 14: 'o', 15: 'A', 16: 'n', 17: 'e', 18: 'y', 19: 'C', 20: '\n', 21: 'd', 22: 'S', 23: '.', 24: 'p', 25: ' ', 26: 'w', 27: 'u', 28: 'z', 29: 'k', 30: 'B'}
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [ ]:
class RNN:
    def __init__(self, hidden_size, vocab_size, seq_length, learning_rate):

        # hyperparameters 저장. hidden_size는 RNN의 기억용량(은닉층 뉴런 개수), vocab_size는 문자 종류 수(입출력 벡터 차원).
        self.hidden_size = hidden_size   # size of hidden layer of neurons
        self.vocab_size = vocab_size
        self.seq_length = seq_length     # number of steps to unroll the RNN for
        self.learning_rate = learning_rate

        # model parameters(RNN의 4가지 핵심 가중치)
        self.U = np.random.uniform(-np.sqrt(1. / vocab_size), np.sqrt(1. / vocab_size), (hidden_size, vocab_size)) # U:입력(문자)-> 은닉층으로 가는 가중치
        self.V = np.random.uniform(-np.sqrt(1. / hidden_size), np.sqrt(1. / hidden_size), (vocab_size, hidden_size)) # V:은닉층 -> 출력(다음 문자 예측)으로 가는 가중치
        self.W = np.random.uniform(-np.sqrt(1. / hidden_size), np.sqrt(1. / hidden_size), (hidden_size, hidden_size)) # W:이전 은닉 상태 -> 현재 은닉 상태로 가는 가중치(이게 "기억"을 만드는 부분)
        self.b = np.zeros((hidden_size, 1))  # bias for hidden layer 은닉층 편향
        self.c = np.zeros((vocab_size, 1))  # bias for output 출력층 편향

        self.mU = np.zeros_like(self.U) # m 붙은 것들은 Adagrad 옵티마이저용 누적 제곱합 메모리. 나중에 update_model에서 씀.
        self.mW = np.zeros_like(self.W)
        self.mV = np.zeros_like(self.V)
        self.mb = np.zeros_like(self.b)
        self.mc = np.zeros_like(self.c)

    def softmax(self, x): # 출력값을 확률분포로 바꿔주는 함수.
        p = np.exp(x- np.max(x)) # x-np.max(x)를 먼저 빼주는 건 overflow 방지 때문. 큰 수를 그냥 exp()에 넣으면 값이 무한대로 튈 수 있기 때문.
        return p / np.sum(p)

    def forward(self, inputs, hprev): # inputs는 정수 인덱스 시퀀스. hprev는 이전 배치에서 넘어온 은닉 상태. hs[-1]에 이걸 저장해서, t=0일 때 "이전 상태"로 참조할 수 있게 함.
        xs, hs, os, ycap = {}, {}, {}, {} # xs:입력문자, hs:은닉상태, os:점수(정규화 안 됨), ycap:확률분포
        hs[-1] = np.copy(hprev)

        for t in range(len(inputs)): # 각 시점 t마다, 정수 인덱스를 원-핫 벡터로 변환.
            xs[t] = np.zeros((self.vocab_size, 1))
            xs[t][inputs[t]] = 1  # one hot encoding , 1-of-k
            hs[t] = np.tanh(np.dot(self.U, xs[t]) + np.dot(self.W, hs[t - 1]) + self.b)  # hidden state update 공식
            os[t] = np.dot(self.V, hs[t]) + self.c  # 은닉 상태에서 다음 글자에 대한 os[t]를 뽑고, softmax로 확률분포 ycap[t]로 변환. 이게 "다음 글자가 뭘까"에 대한 모델의 예측.
            ycap[t] = self.softmax(os[t])  # next char에 대한 softmax 확률값 계산
        return xs, hs, ycap # 나중에 backward에서 다 필요하니까 딕셔너리 형태로 전부 저장해서 반환.


    def backward(self, xs, hs, ps, targets):
        # backward pass: 기울기를 역방향으로 계산하는 과정
        dU, dW, dV = np.zeros_like(self.U), np.zeros_like(self.W), np.zeros_like(self.V) # 각 파라미터의 기울기를 담을 그릇들을 0으로 초기화.
        db, dc = np.zeros_like(self.b), np.zeros_like(self.c)

        dhnext = np.zeros_like(hs[0]) # dhnext는 "미래 시점에서 흘러들어오는 기울기"를 담는 변수(처음엔 없으니 0)
        for t in reversed(range(self.seq_length)): # 시간을 거꾸로 돌면서 역전파. RNN은 시간축으로 펼쳐놓은 구조라서, 마지막 시점부터 거꾸로 기울기를 계산해가야함. = BPTT
            dy = np.copy(ps[t]) # Softmax + Cross-entropy loss를 함께 미분하면 공식이 깔끔.
            dy[targets[t]] -= 1  # 정답 인덱스 위치만 1 빼줌. 이게 출력층에서의 기울기 dy.

            dV += np.dot(dy, hs[t].T) # calculate dV, dc
            dc += dy

            dh = np.dot(self.V.T, dy) + dhnext  # 은닉 상태 h_t가 영향을 미치는 경로가 두 갈래. 출력(V)쪽으로, 다음 시점의 은닉상태로. 그래서 두 기울기를 더해줘야함.(dhnext)가 2번 경로에서 넘어온것.

            dhrec = (1 - hs[t] * hs[t]) * dh # 비선형 tanh 활성화함수를 통한 역전파

            db += dhrec # U,W,b는 은닉층 계산에 직접 관여하니까 dhrec(tanh 통과 후 기울기)를 이용해 계산.
            dU += np.dot(dhrec, xs[t].T)
            dW += np.dot(dhrec, hs[t - 1].T)

            dhnext = np.dot(self.W.T, dhrec) # 이번 시점의 기울기 중 "이전 시점으로 흘러가야 할 몫"을 계산해서 다음 루프에서 쓰게 넘겨줌.

        for dparam in [dU, dW, dV, db, dc]: # Gradent clipping. RNN은 시퀀스가 길어지면 기울기가 기하급수적으로 커지는 "exploding gradient"문제가 흔한데, 값이 -5~5 범위를 벗어나면 강제로 잘라내서 학습이 발산하는걸 막아줌.
            np.clip(dparam, -5, 5, out=dparam)
        return dU, dW, dV, db, dc

    def loss(self, ps, targets):
        """loss for a sequence"""
        # calculate cross-entrpy loss
        return sum(-np.log(ps[t][targets[t], 0]) for t in range(self.seq_length)) # Cross-entropy loss. 정답 문자에 모델이 부여한 확률이 높을수록 loss가 작음. 시퀀스 전체 시점에 대해 다 더함.

    def update_model(self, dU, dW, dV, db, dc):
        # parameter update with adagrad
        for param, dparam, mem in zip([self.U, self.W, self.V, self.b, self.c],
                                      [dU, dW, dV, db, dc],
                                      [self.mU, self.mW, self.mV, self.mb, self.mc]):
            mem += dparam * dparam  # mem = mem + (dparam * dparam) # Adagrad: 각 파라미터별로 지금까지 나온 기울기 제곱을 누적해두고, 그 누적값의 제곱근으로 학습률을 나눠줌. 자주 업데이트된 파라미터는 점점 학습률이 작아지고, 드물게 업데이트된 파라미터는 상대적으로 큰 스텝을 유지.
            param += -self.learning_rate * dparam / np.sqrt(mem + 1e-8)  # adagrad update

    def sample(self, h, seed_ix, n): # 학습 도중 "지금 모델이 얼마나 그럴듯한 텍스트를 만드나" 확인용.
        """
        정수 형태의 sequence 데이터 샘플링 과정
        h 는 메모리 state, seed_ix 는 첫 time step의 시드 문자
        """
        x = np.zeros((self.vocab_size, 1))
        x[seed_ix] = 1
        ixes = []
        for t in range(n):
            h = np.tanh(np.dot(self.U, x) + np.dot(self.W, h) + self.b)
            y = np.dot(self.V, h) + self.c
            p = np.exp(y) / np.sum(np.exp(y))
            ix = np.random.choice(range(self.vocab_size), p=p.ravel()) # 확률에 따라 무작위 샘플링
            x = np.zeros((self.vocab_size, 1))
            x[ix] = 1
            ixes.append(ix)
        return ixes

    def train(self, data_reader):
        iter_num = 0
        threshold = 0.01
        smooth_loss = -np.log(1.0 / data_reader.vocab_size) * self.seq_length
        while (smooth_loss > threshold): # 데이터를 처음부터 다시 읽기 시작하는 시점. pointer==0이면 은닉상태를 리셋. 그 외에는 이전 배치의 hprev를 이어서 씀.
            if data_reader.just_started():
                hprev = np.zeros((self.hidden_size, 1))

            inputs, targets = data_reader.next_batch() # 한 배치당 순전파 -> 역전파 -> loss계산 -> 파라미터 업데이트 -> smooth_loss 갱신 -> 다음 배치를 위해 마지막 은닉 상태를 hprev로 저장.
            xs, hs, ps = self.forward(inputs, hprev)
            dU, dW, dV, db, dc = self.backward(xs, hs, ps, targets)
            loss = self.loss(ps, targets)
            self.update_model(dU, dW, dV, db, dc)
            smooth_loss = smooth_loss * 0.999 + loss * 0.001
            hprev = hs[self.seq_length - 1]

            if not iter_num % 500: # 500번마다 한 번씩 샘플 텍스트를 찍어서 학습이 잘 되고 있는지 눈으로 확인
                sample_ix = self.sample(hprev, inputs[0], 200)
                print(''.join(data_reader.ix_to_char[ix] for ix in sample_ix))
                print("\n\niter :%d, loss:%f" % (iter_num, smooth_loss))
            iter_num += 1

    def predict(self, data_reader, start, n):

        # 입력 벡터 초기화
        x = np.zeros((self.vocab_size, 1))
        chars = [ch for ch in start]
        ixes = []
        for i in range(len(chars)):
            ix = data_reader.char_to_ix[chars[i]]
            x[ix] = 1
            ixes.append(ix)

        h = np.zeros((self.hidden_size, 1))
        # 다음에 올 문자 n 예측
        for t in range(n):
            h = np.tanh(np.dot(self.U, x) + np.dot(self.W, h) + self.b)
            y = np.dot(self.V, h) + self.c
            p = np.exp(y) / np.sum(np.exp(y))
            ix = np.random.choice(range(self.vocab_size), p=p.ravel())
            x = np.zeros((self.vocab_size, 1))
            x[ix] = 1
            ixes.append(ix)
        txt = ''.join(data_reader.ix_to_char[i] for i in ixes)
        return txt


In [ ]:
seq_length = 50
#read text from the "input.txt" file
data_reader = DataReader("input.txt", seq_length)
rnn = RNN(hidden_size=100, vocab_size=data_reader.vocab_size, seq_length=seq_length, learning_rate=1e-1)
rnn.train(data_reader)

self.char_to_ix: {'F': 0, 's': 1, 't': 2, 'h': 3, 'm': 4, 'r': 5, ',': 6, 'c': 7, ':': 8, 'f': 9, 'l': 10, 'Y': 11, 'i': 12, 'a': 13, 'o': 14, 'A': 15, 'n': 16, 'e': 17, 'y': 18, 'C': 19, '\n': 20, 'd': 21, 'S': 22, '.': 23, 'p': 24, ' ': 25, 'w': 26, 'u': 27, 'z': 28, 'k': 29, 'B': 30}
self.ix_to_char: {0: 'F', 1: 's', 2: 't', 3: 'h', 4: 'm', 5: 'r', 6: ',', 7: 'c', 8: ':', 9: 'f', 10: 'l', 11: 'Y', 12: 'i', 13: 'a', 14: 'o', 15: 'A', 16: 'n', 17: 'e', 18: 'y', 19: 'C', 20: '\n', 21: 'd', 22: 'S', 23: '.', 24: 'p', 25: ' ', 26: 'w', 27: 'u', 28: 'z', 29: 'k', 30: 'B'}
rkh:zto ftieSe fBm y,phrk
poAeawCf r,pea,p:o teweeri
ti: aoyrd.mCrrAnzeeF CiookerAiAe StAwp e,tinrtptcaSzterckoewtymsup 
fmp,uArrerdacse
,oAm iuerAeB,pSeirlue,on,eul tCieduBh
  nlpo pi wu enhy .cytaih


iter :0, loss:171.699639
any fuather, hear any further, hear any further, hear any further, hear any further, hear any furthert hear any further, hear any further, heed any further, hear any further, hear any further, hea

In [ ]:
rnn.predict(data_reader, 'test', 50)

'test wizen:\nBefore we proceed any further, hear any fu'

In [ ]:
rnn.predict(data_reader, 'we', 50)

'wee Beaear any further, hear any further, hear any f'

### 다양한 방식으로 text에 입력을 주어 모델 학습 및 그 결과 확인해보기
